In [14]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, brier_score_loss
import numpy as np
import seaborn as sns

In [15]:
df = pd.read_csv("games.csv")

In [18]:
y = (df["win"].astype(str).str.lower() == "true").astype(int).values

In [19]:
features = [
    "kills", "deaths", "assists",
    "experience", "gold", "visionScore",
    "damageDealtToChampions", "damageDealtToBuildings", "damageDealtToObjectives",
    "totalHealsOnTeammates", "totalDamageShieldedOnTeammates", "timeCCingOthers",
]
X = df[features].values

In [20]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, solver="lbfgs"),
)
model.fit(X, y)

/home/fahd/.pyenv/versions/3.12.2/envs/lolenv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression', LogisticRegression(max_iter=1000))])

In [21]:
df["pred_win"] = model.predict_proba(X)[:, 1]

In [22]:
df.head()

,matchid,rank,championName,role,side,damageDealtToChampions,damageDealtToBuildings,damageDealtToObjectives,kills,deaths,...,level,experience,gold,summonerLevel,visionScore,totalHealsOnTeammates,totalDamageShieldedOnTeammates,timeCCingOthers,win,pred_win
0,EUW1_8021537421,MASTER,Lulu,TOP,blue,8572,146,146,0,7,...,13,10348,5973,68,19,0,3216,40,False,0.418040
1,EUW1_8021537421,MASTER,Viego,JUNGLE,blue,19973,0,6093,8,10,...,14,11849,11574,123,12,0,0,11,False,0.359497
2,EUW1_8021537421,MASTER,Fizz,MIDDLE,blue,20149,11906,11906,9,8,...,14,12886,12207,55,23,0,0,12,False,0.210026
3,EUW1_8021537421,MASTER,Kaisa,BOTTOM,blue,17432,6063,6063,10,7,...,13,10981,13726,827,11,0,0,1,False,0.466116
4,EUW1_8021537421,MASTER,Leona,UTILITY,blue,8376,1357,1357,1,10,...,11,8494,7700,49,53,105,140,44,False,0.354555


In [23]:
pred = model.predict_proba(X)[:, 1]
pred_label = (pred >= 0.5).astype(int)

print("accuracy :", accuracy_score(y, pred_label))
print("log_loss :", log_loss(y, pred))
print("auc      :", roc_auc_score(y, pred))
print("brier    :", brier_score_loss(y, pred))

# Expected Calibration Error (ECE)
def ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    idx  = np.digitize(y_prob, bins) - 1
    total = len(y_true)
    e = 0.0
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        conf = y_prob[mask].mean()
        acc  = y_true[mask].mean()
        e += (mask.sum() / total) * abs(acc - conf)
    return e

print("ece      :", ece(y, pred))

accuracy : 0.7652728157846957
log_loss : 0.5025165790242545
auc      : 0.8380433549384303
brier    : 0.16379547370700065
ece      : 0.024118930203387668


In [24]:
overall = df.groupby("championName")["pred_win"].mean()
print(overall.sort_values(ascending=False).head(20))

championName
Yuumi       0.687633
Milio       0.652518
Ivern       0.649773
Janna       0.641263
Nami        0.637631
Zilean      0.636102
JarvanIV    0.631693
Rell        0.620796
Rakan       0.616863
Sejuani     0.616031
Rammus      0.614380
Lulu        0.613263
Zac         0.609253
Bard        0.607733
Sona        0.602641
RekSai      0.598468
Skarner     0.597229
Renata      0.589182
Braum       0.588998
Soraka      0.587214
Name: pred_win, dtype: float64


In [25]:
x = df[df['championName'] == 'Kassadin']['pred_win'].values

In [ ]:
rosters = (
    df.groupby(["matchid", "side"])["championName"]
      .apply(list)
      .unstack("side")
      .rename(columns={"blue": "blue_team", "red": "red_team"})
)
df = df.merge(rosters, left_on="matchid", right_index=True, how="left")
df["enemy_team"] = np.where(df["side"] == "blue", df["red_team"], df["blue_team"])

# ── 2. Baseline: mean pred_win per champion over ALL their games ─────
baseline = (
    df.groupby("championName")["pred_win"]
      .agg(baseline_sum="sum", baseline_count="count")
)

# ── 3. Explode into (A, B) pairs, one per enemy champion per row ─────
pairs = df[["championName", "pred_win", "enemy_team"]].explode("enemy_team")
pairs = pairs.rename(columns={"championName": "A", "enemy_team": "B"})
pair_stats = (
    pairs.groupby(["A", "B"])["pred_win"]
         .agg(matchup_sum="sum", matchup_count="count")
         .reset_index()
)
pair_stats["matchup_mean"] = pair_stats["matchup_sum"] / pair_stats["matchup_count"]

# ── 4. Baseline excluding B, then strength = baseline_excl - matchup ─
pair_stats = pair_stats.merge(baseline, left_on="A", right_index=True)
pair_stats["baseline_excl_mean"] = (
    (pair_stats["baseline_sum"] - pair_stats["matchup_sum"]) /
    (pair_stats["baseline_count"] - pair_stats["matchup_count"])
)
pair_stats["strength"] = (
    pair_stats["baseline_excl_mean"] - pair_stats["matchup_mean"]
)

# ── 5. Pivot to 173 x 173 (A rows, B columns) ────────────────────────
champs = sorted(df["championName"].unique())
strength = (
    pair_stats.pivot(index="A", columns="B", values="strength")
              .reindex(index=champs, columns=champs)
)

strength.to_csv("matchup_strengths.csv")
print(f"Saved {strength.shape[0]} x {strength.shape[1]} matrix "
      f"({strength.notna().sum().sum()} populated cells).")

In [ ]:
df = pd.read_csv('matchup_strengths.csv')

In [ ]:
df = df.set_index('A')

In [ ]:
df['Yone']['Annie']